<a href="https://colab.research.google.com/github/kjahan/armory/blob/main/notebooks/t5_paraphraser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paraphrase-Generation

Model: `Vamsi/T5_Paraphrase_Paws`

Ref: https://huggingface.co/Vamsi/T5_Paraphrase_Paws


## Install transformers with sentencepiece

We ran into the following issue so we need to install `sentencepiece` to resolve it:

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece installed to convert a slow tokenizer to a fast one.



In [1]:
!pip install transformers[sentencepiece]

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 4.4 MB 8.4 MB/s 
     |████████████████████████████████| 86 kB 4.0 MB/s 
     |████████████████████████████████| 6.6 MB 58.6 MB/s 
     |████████████████████████████████| 596 kB 74.0 MB/s 
     |████████████████████████████████| 1.2 MB 46.7 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


## Imports

### Paraphrase-Generation
​

### Model description
​T5 Model for generating paraphrases of english sentences. Trained on the Google PAWS dataset.​

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## Setup tokenizer/model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws")  
model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws").to("cuda")

Downloading:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/773k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/850M [00:00<?, ?B/s]

## Your input sentence for paraphrasing

In [ ]:
# sentence = "The spread of Omicron has roiled financial markets and prompted governments around the world to tighten travel and workplace restrictions."
# sentence = "The Federal Reserve has spent most of 2021 saying that high inflation would be temporary."
sentence = "The good news is that the Fed can taper fast enough to let it raise interest rates in March."

text =  "paraphrase: " + sentence + " </s>"

## Prepare inputs

Make sure you change your `Runtime` to run this notebook on a GPU; otherwise, we will run into the following error:

`RuntimeError: No CUDA GPUs are available`

In [ ]:
encoding = tokenizer.encode_plus(text, pad_to_max_length=True, return_tensors="pt")
input_ids, attention_masks = encoding["input_ids"].to("cuda"), encoding["attention_mask"].to("cuda")

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2291: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,


## Generate output

We had to make sure to load the model in GPU with the following command:

`model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws").to("cuda")`

Otherwise, we would run to a RunTime error as shown below:

`RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper__index_select)`

Refrence:
https://discuss.huggingface.co/t/runtimeerror-expected-all-tensors-to-be-on-the-same-device-but-found-at-least-two-devices-cpu-and-cuda-0-when-checking-arugment-for-argument-index-in-method-wrapper-index-select/9255

In [ ]:
outputs = model.generate(
    input_ids=input_ids, attention_mask=attention_masks,
    max_length=256,
    do_sample=True,
    top_k=120,
    top_p=0.95,
    early_stopping=True,
    num_return_sequences=5
)

## Print paraphrased output

`"The spread of Omicron has roiled financial markets and prompted governments around the world to tighten travel and workplace restrictions."
`

In [ ]:
for output in outputs:
    line = tokenizer.decode(output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print(line)

Good news is that the Fed can trim fast enough to raise interest rates in March.
The good news is that the Fed can deflationize quickly enough to allow it to raise interest rates in March.
The good news is that the Fed can reduce interest rates fast enough to let it raise them in March.
The good news is that the Fed can rise rates quickly enough to raise interest rates in March.
The good news is that the Fed can taper in March enough to raise the interest rates.


## Paraphrased

`sentence = "The Federal Reserve has spent most of 2021 saying that high inflation would be temporary."`


In [ ]:
print("Original paragraph:")
print("{}\n".format(sentence))

for output in outputs:
    line = tokenizer.decode(output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print(line)

Original paragraph:
The good news is that the Fed can taper fast enough to let it raise interest rates in March.

Good news is that the Fed can trim fast enough to raise interest rates in March.
The good news is that the Fed can deflationize quickly enough to allow it to raise interest rates in March.
The good news is that the Fed can reduce interest rates fast enough to let it raise them in March.
The good news is that the Fed can rise rates quickly enough to raise interest rates in March.
The good news is that the Fed can taper in March enough to raise the interest rates.


## Generate paraphrase

`sentence = "The good news is that the Fed can taper fast enough to let it raise interest rates in March."`

In [ ]:
print("Original paragraph:")
print("{}\n".format(sentence))

for output in outputs:
    line = tokenizer.decode(output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print(line)

Original paragraph:
The good news is that the Fed can taper fast enough to let it raise interest rates in March.

Good news is that the Fed can trim fast enough to raise interest rates in March.
The good news is that the Fed can deflationize quickly enough to allow it to raise interest rates in March.
The good news is that the Fed can reduce interest rates fast enough to let it raise them in March.
The good news is that the Fed can rise rates quickly enough to raise interest rates in March.
The good news is that the Fed can taper in March enough to raise the interest rates.


## Let's compare T5 vs QuillBot

Input:
`Kazem has grown up in Iran where he did his undergrad in Electrical Engineering and his master's in distributed computing. He moved to Canada to start his Ph.D. in Computer Science. Since 2012, Kazem has worked for a number of startups where a few found product-market fit & scaled while others were not so lucky! In 2014, Kazem co-founded an ad-tech startup. Over the last few years, Kazem has worked as a lead ML/NLP engineer at Match Group (PoF/Tiner/Match), Tableau, Salesforce, and Course Hero. Kazem spends his free time swimming, biking, hostage chess, and playing poker.`

QuillBot output:
`
Kazem grew raised in Iran, where he earned his bachelor's degree in electrical engineering and his master's degree in distributed computing. He relocated to Canada to pursue a Ph.D. in computer science. Kazem has worked for a variety of businesses since 2012, some of which achieved product-market fit and scaled, but others did not! Kazem co-founded an ad-tech firm in 2014. Kazem has worked as a top ML/NLP developer at Match Group (PoF/Tiner/Match), Tableau, Salesforce, and Course Hero in recent years. Kazem enjoys swimming, bicycling, hostage chess, and poker in his spare time.`

Ref: https://quillbot.com/

In [ ]:
sentence = "Kazem has grown up in Iran where he did his undergrad in Electrical Engineering and his master's in distributed computing. He moved to Canada to start his Ph.D. in Computer Science."

#  Since 2012, Kazem has worked for a number of startups where a few found product-market fit & scaled while others were not so lucky! In 2014, Kazem co-founded an ad-tech startup. Over the last few years, Kazem has worked as a lead ML/NLP engineer at Match Group (PoF/Tiner/Match), Tableau, Salesforce, and Course Hero. Kazem spends his free time swimming, biking, hostage chess, and playing poker.

text =  "paraphrase: " + sentence + " </s>"

# Prepare input
encoding = tokenizer.encode_plus(text, pad_to_max_length=True, return_tensors="pt")
input_ids, attention_masks = encoding["input_ids"].to("cuda"), encoding["attention_mask"].to("cuda")

outputs = model.generate(
    input_ids=input_ids, 
    attention_mask=attention_masks,
    max_length=256,
    do_sample=True,
    top_k=120,
    top_p=0.98,
    early_stopping=True,
    num_return_sequences=5
)

for output in outputs:
    line = tokenizer.decode(output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print("{}\n".format(line))

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2291: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,


Kazem has grown up in Iran where he did his undergrad in electrical engineering and taught his master degree in distributed computing, and he moved to Canada to start his Ph.D. in Computer Science.

Kazem has grown up in Iran where he did his master thesis in electronics engineering and his undergraduate in distributed computing and moved to Canada to set his Ph.D. in computer science.

Kazem grew up in Iran where he did his undergrad in electrical engineering and his master's in distributed computing and moved to Canada to begin his Ph.D. in Computer Science.

Kazem has grown up in Iran, where he did his undergraduate work in electrical engineering and his master's in distributed computing, and moved to Canada to begin his Ph.D. in computer science.

Kazem has grown up in Iran, where he did his undergrad work in electrical engineering and his master in distributed computing and moved to Canada to start his Ph.D. in Computer Science.



## Paraphrase any question with T5 (Text-To-Text Transfer Transformer) — Pretrained model and training script provided

https://towardsdatascience.com/paraphrase-any-question-with-t5-text-to-text-transfer-transformer-pretrained-model-and-cbb9e35f1555

In [3]:
import torch
from transformers import T5ForConditionalGeneration,T5Tokenizer


def set_seed(seed):
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(42)

model_2 = T5ForConditionalGeneration.from_pretrained('ramsrigouthamg/t5_paraphraser')
tokenizer_2 = T5Tokenizer.from_pretrained('ramsrigouthamg/t5_paraphraser')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("device ",device)
model_2 = model_2.to(device)

device  cuda


In [9]:
sentence = "Blockchain technologies are quickly gaining popularity and enable the borderless and frictionless transfer of assets. In this course, we will study blockchain fundamentals and applications built on top of them. Topics include signature schemes, commitment schemes, multi-party computation, zero-knowledge proofs, consensus mechanisms, smart contracts, incentive mechanisms, applications of blockchains such as cryptocurrencies, and the legal framework around them."
# sentence = "Gold is just bitcoin that can't be sent over the internet."
# sentence = "Which investment is better during stagflation: stocks, gold, commidity, bitcoin or real estate?"
# sentence = "Which course should I take to get started in data science?"
# sentence = "What are the ingredients required to bake a perfect cake?"
# sentence = "What is the best possible approach to learn aeronautical engineering?"
# sentence = "Do apples taste better than oranges in general?"


text =  "paraphrase: " + sentence # + " </s>"


max_len_val = 512

encoding = tokenizer_2.encode_plus(text, padding=True, return_tensors="pt")
input_ids, attention_masks = encoding["input_ids"].to(device), encoding["attention_mask"].to(device)


# set top_k = 50 and set top_p = 0.95 and num_return_sequences = 3
beam_outputs = model_2.generate(
    input_ids=input_ids, 
    attention_mask=attention_masks,
    do_sample=True,
    max_length=max_len_val,
    top_k=120,
    top_p=0.98,
    early_stopping=True,
    num_return_sequences=5
)


print ("\nOriginal Question ::")
print (sentence)
print ("\n")
print ("Paraphrased Questions :: ")
final_outputs =[]
for beam_output in beam_outputs:
    sent = tokenizer_2.decode(beam_output, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    if sent.lower() != sentence.lower() and sent not in final_outputs:
        final_outputs.append(sent)

for i, final_output in enumerate(final_outputs):
    print("{}: {}".format(i, final_output))


Original Question ::
Blockchain technologies are quickly gaining popularity and enable the borderless and frictionless transfer of assets. In this course, we will study blockchain fundamentals and applications built on top of them. Topics include signature schemes, commitment schemes, multi-party computation, zero-knowledge proofs, consensus mechanisms, smart contracts, incentive mechanisms, applications of blockchains such as cryptocurrencies, and the legal framework around them.


Paraphrased Questions :: 
0: In this course, we will study blockchain fundamentals and applications built on top of them. Topics include signature schemes, commitment schemes, multi-party computation, zero-knowledge proofs, consensus mechanisms, smart contracts, incentive mechanisms, applications of blockchains such as cryptocurrencies, and the legal framework around them.
1: Blockchain technologies enabled by the interdependence of citizens. In this course, we will study fundamentals of blockchain such as